# Exploración de datos: inasistencia a turnos médicos

## 1. Objetivo del análisis

Este notebook tiene como objetivo explorar el dataset *Medical Appointment No Shows* para evaluar su utilidad en un proyecto de ingeniería de features mediante SQL analítico.

La pregunta de negocio es:

**¿Podemos identificar patrones históricos de comportamiento que ayuden a estimar la probabilidad de que un paciente falte a su próximo turno médico?**

Para responderla, se analizarán la estructura del dataset, la calidad de los datos, las fechas de las citas, la frecuencia de visitas por paciente y la distribución de las inasistencias.

Los resultados de esta exploración permitirán definir las variables predictivas que se construirán posteriormente con SQL y DuckDB.

## 2. Fuente de datos

* **Dataset:** Medical Appointment No Shows.
* **Fuente:** [Kaggle — Medical Appointment No Shows](https://www.kaggle.com/datasets/joniarroba/noshowappointments).
* **Unidad de observación:** una cita médica programada.
* **Variable de resultado:** `No-show`, que indica si el paciente faltó a la cita.

## 3. Alcance y consideraciones

Este proyecto estudia la inasistencia a citas médicas, no el abandono de un tratamiento clínico.

Las variables predictivas deberán construirse exclusivamente con información disponible antes de la cita que se desea predecir, para evitar la fuga de información (*data leakage*).


In [6]:
from pathlib import Path
import pandas as pd

# Directorio raíz del proyecto
PROJECT_DIR = Path.cwd().parent

# Ruta del dataset
ruta = PROJECT_DIR / "data" / "KaggleV2-May-2016.csv"

# Cargar los datos
df = pd.read_csv(ruta)

print("Dataset cargado correctamente.")
display(df.head())

Dataset cargado correctamente.


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


## 4. Exploración estructural

En esta sección se revisan las dimensiones del dataset, los nombres de las variables y sus tipos de datos para comprender la estructura inicial de la información.


In [2]:
print("Cantidad de filas y columnas:", df.shape)

print("\nNombres de las columnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
display(df.dtypes)

Cantidad de filas y columnas: (110527, 14)

Nombres de las columnas:
['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show']

Tipos de datos:


PatientId         float64
AppointmentID       int64
Gender                str
ScheduledDay          str
AppointmentDay        str
Age                 int64
Neighbourhood         str
Scholarship         int64
Hipertension        int64
Diabetes            int64
Alcoholism          int64
Handcap             int64
SMS_received        int64
No-show               str
dtype: object

## 5. Frecuencia de turnos por paciente

Se analiza cuántos turnos tiene registrados cada paciente y cuántos pacientes presentan visitas repetidas. Esta comprobación es fundamental para determinar si pueden construirse variables históricas de comportamiento, como la frecuencia de visitas y la tasa de inasistencia.

El dataset contiene 62.299 pacientes únicos, de los cuales 24.379 tienen más de un turno registrado.


In [3]:
# Cantidad de turnos por paciente
turnos_por_paciente = df.groupby("PatientId").size()

print("Pacientes únicos:", df["PatientId"].nunique())
print("Pacientes con más de un turno:", (turnos_por_paciente > 1).sum())
print("Promedio de turnos por paciente:", round(turnos_por_paciente.mean(), 2))

print("\nDistribución de cantidad de turnos:")
display(turnos_por_paciente.describe())

Pacientes únicos: 62299
Pacientes con más de un turno: 24379
Promedio de turnos por paciente: 1.77

Distribución de cantidad de turnos:


count    62299.000000
mean         1.774138
std          1.770324
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         88.000000
dtype: float64

## 6. Análisis temporal y validación de fechas

Se convierten las columnas de fechas a un formato temporal para evaluar el período cubierto por los registros e identificar posibles inconsistencias.

Se comparan las fechas de programación con las fechas de las citas para detectar registros que requieran revisión antes de generar variables temporales.


In [6]:
# Convertir las fechas y quitar la zona horaria
# para trabajar con fechas locales sin información de hora UTC.

for columna in ["ScheduledDay", "AppointmentDay"]:
    df[columna] = pd.to_datetime(
        df[columna],
        errors="coerce",
        utc=True
    ).dt.tz_localize(None)

print("Rango de fechas:")
display(df[["ScheduledDay", "AppointmentDay"]].agg(["min", "max"]))

print("\nFechas inválidas:")
display(df[["ScheduledDay", "AppointmentDay"]].isna().sum())

print("\nCitas programadas antes de la fecha de programación:")
print((df["AppointmentDay"] < df["ScheduledDay"].dt.normalize()).sum())

Rango de fechas:


,ScheduledDay,AppointmentDay
min,2015-11-10 07:13:56,2016-04-29
max,2016-06-08 20:07:23,2016-06-08



Fechas inválidas:


ScheduledDay      0
AppointmentDay    0
dtype: int64


Citas programadas antes de la fecha de programación:
5


## 7. Calidad de datos y distribución de inasistencias

Se verifica la unicidad del identificador de cada turno y se analiza la distribución de la variable `No-show`.

Esta variable será el objetivo potencial de predicción: indica si el paciente asistió o no a la cita programada.


In [7]:
# Verificar duplicados en el identificador de turno
print(
    "AppointmentID duplicados:",
    df["AppointmentID"].duplicated().sum()
)

# Revisar valores de la variable objetivo
print("\nDistribución de No-show:")
display(df["No-show"].value_counts(dropna=False))

# Calcular el porcentaje de inasistencias
porcentaje_inasistencias = (
    df["No-show"].eq("Yes").mean() * 100
)

print(
    "\nPorcentaje de inasistencias:",
    round(porcentaje_inasistencias, 2),
    "%"
)

AppointmentID duplicados: 0

Distribución de No-show:


No-show
No     88208
Yes    22319
Name: count, dtype: int64


Porcentaje de inasistencias: 20.19 %


## 8. Análisis de frecuencia de visitas

Se analiza cuántos pacientes tienen registros de dos, tres o cinco turnos como mínimo. También se examina la distribución de citas por paciente para evaluar si existe suficiente historial para generar variables de comportamiento.


In [8]:
# Contar turnos por paciente
turnos = df.groupby("PatientId").size()

print("Pacientes con al menos 2 turnos:", (turnos >= 2).sum())
print("Pacientes con al menos 3 turnos:", (turnos >= 3).sum())
print("Pacientes con al menos 5 turnos:", (turnos >= 5).sum())

# Distribución de turnos por paciente
display(turnos.value_counts().sort_index().head(10))

Pacientes con al menos 2 turnos: 24379
Pacientes con al menos 3 turnos: 10484
Pacientes con al menos 5 turnos: 2617


1     37920
2     13895
3      5500
4      2367
5      1119
6       553
7       306
8       202
9       104
10       85
Name: count, dtype: int64

In [10]:
print(df["AppointmentDay"].dtype)
display(df["AppointmentDay"].head())

str


0    2016-04-29T00:00:00Z
1    2016-04-29T00:00:00Z
2    2016-04-29T00:00:00Z
3    2016-04-29T00:00:00Z
4    2016-04-29T00:00:00Z
Name: AppointmentDay, dtype: str

In [11]:
df["AppointmentDay"] = pd.to_datetime(
    df["AppointmentDay"],
    errors="coerce",
    utc=True
).dt.tz_localize(None)

print("Tipo de dato:", df["AppointmentDay"].dtype)

Tipo de dato: datetime64[us]


## 9. Distribución temporal de las citas

Se analiza la cantidad de citas médicas registradas por mes para conocer la cobertura temporal del dataset e identificar cómo se distribuyen los turnos durante el período estudiado.

Esta información es relevante para la ingeniería de features, ya que las variables de comportamiento se construirán a partir del historial disponible antes de cada cita.


In [12]:
# Cantidad de citas por mes
citas_por_mes = (
    df.groupby(df["AppointmentDay"].dt.to_period("M"))
      .size()
      .rename("cantidad_citas")
)

display(citas_por_mes)

AppointmentDay
2016-04     3235
2016-05    80841
2016-06    26451
Freq: M, Name: cantidad_citas, dtype: int64

## **Interpretación de los resultados**

La mayor cantidad de citas corresponde a mayo de 2016, con 80.841 registros. En abril se registraron 3.235 citas y en junio, 26.451.

La distribución temporal está concentrada en un período relativamente corto. Esta limitación deberá tenerse en cuenta al construir variables históricas y evaluar qué información estaba disponible antes de cada turno.


## 10. Conclusiones de la exploración

### Calidad y estructura de los datos

El dataset contiene 110.527 registros de citas médicas y 62.299 pacientes únicos. El identificador `AppointmentID` no presenta duplicados, por lo que cada registro corresponde a una cita diferenciada.

Se identificaron 24.379 pacientes con más de un turno. Esto permite analizar patrones históricos de comportamiento, aunque no todos los pacientes cuentan con suficiente historial para calcular variables temporales.

### Comportamiento de las inasistencias

La variable `No-show` registra 22.319 inasistencias, equivalentes al 20,19 % de las citas. Este valor representa la proporción general de inasistencias del dataset y servirá como referencia para interpretar las variables que se construyan.

### Cobertura temporal y calidad de las fechas

Las fechas de las citas se concentran entre abril y junio de 2016. Además, se detectaron cinco registros en los que la fecha de la cita es anterior a la fecha de programación. Estos casos deberán revisarse y tratarse explícitamente durante la preparación de los datos.

### Variables candidatas para la ingeniería de features

A partir de la exploración, se propone construir las siguientes variables mediante SQL analítico:

* **Recencia:** días transcurridos desde la cita anterior del paciente.
* **Frecuencia histórica:** cantidad de citas anteriores a la cita evaluada.
* **Tasa histórica de inasistencia:** proporción de citas anteriores en las que el paciente no asistió.
* **Tendencia de visitas:** comparación de la actividad reciente del paciente con la de un período anterior.

Estas variables permitirán estudiar si el historial de asistencia aporta información útil para estimar futuras inasistencias.

### Consideraciones metodológicas

Las variables predictivas deberán calcularse utilizando únicamente información disponible antes de cada cita. El resultado `No-show` de la cita evaluada se conservará como variable objetivo, pero no se utilizará para calcular sus propias features.

El proyecto se limita a estudiar la asistencia a citas médicas programadas.